# Ablation Evaluation

This notebook evaluates the four ablation-study classifiers on the three held-out ablation test sets: original, localized, and personified.


## Environment Setup


In [ ]:
# Install only the extra packages needed in Colab.
# Leave Colab's torch / torchvision versions alone.
# Reinstalling one without the other can break CUDA support.
# %pip install -q transformers pandas scikit-learn


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support
from transformers import AutoModelForSequenceClassification, AutoTokenizer


## Project Configuration


In [ ]:
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "datasets"
MODEL_DIR = PROJECT_ROOT / "Abalation Studies Models"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
OUTPUT_DIR = ARTIFACT_DIR / "ablation_evaluation"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not MODEL_DIR.exists():
    raise FileNotFoundError(f"Missing model directory: {MODEL_DIR}")

MODEL_SPECS = {
    "bert_original": {
        "display_name": "BERT + Original",
        "path": MODEL_DIR / "original_bert + original data" / "model",
    },
    "singbert_original": {
        "display_name": "SingBERT + Original",
        "path": MODEL_DIR / "singbert + original data" / "model",
    },
    "singbert_localized": {
        "display_name": "SingBERT + Localized",
        "path": MODEL_DIR / "singbert + localized data" / "model",
    },
    "singbert_personified": {
        "display_name": "SingBERT + Personified",
        "path": MODEL_DIR / "singbert + persona" / "model",
    },
}

TEST_SET_SPECS = {
    "original": {
        "display_name": "Original Test",
        "path": DATA_DIR / "SMSSpamCollection_ablation_test",
        "kind": "tsv",
        "text_column": "text",
    },
    "localized": {
        "display_name": "Localized Test",
        "path": DATA_DIR / "SMSSpamCollection_ablation_test_localized.csv",
        "kind": "csv",
        "text_column": "localized_text",
    },
    "personified": {
        "display_name": "Personified Test",
        "path": DATA_DIR / "SMSSpamCollection_ablation_test_personified.csv",
        "kind": "csv",
        "text_column": "localized_text",
    },
}

CANONICAL_LABEL2ID = {"not_scam": 0, "scam": 1}
CANONICAL_ID2LABEL = {idx: label for label, idx in CANONICAL_LABEL2ID.items()}

MAX_LENGTH = 128
EVAL_BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for model_name, model_spec in MODEL_SPECS.items():
    print(f"{model_name:>20}: {model_spec['path']}")
for test_name, test_spec in TEST_SET_SPECS.items():
    print(f"{test_name:>20}: {test_spec['path']}")
print(f"\nDevice: {DEVICE}")


## Test Set Loading


In [ ]:
def normalize_label_name(label):
    label_text = str(label).strip().lower()
    if label_text in {"ham", "not_scam"}:
        return "not_scam"
    if label_text in {"spam", "scam"}:
        return "scam"
    raise ValueError(f"Unexpected label: {label}")

def load_original_tsv(data_path):
    rows = []
    with data_path.open("r", encoding="utf-8", errors="replace") as fp:
        for line_number, raw_line in enumerate(fp, start=1):
            line = raw_line.rstrip("\n")
            if not line:
                continue
            if "\t" not in line:
                raise ValueError(f"{data_path} line {line_number} is missing a tab separator.")
            raw_label, text = line.split("\t", 1)
            rows.append({"raw_label": raw_label.strip(), "text": text.strip()})
    df = pd.DataFrame(rows)
    df["label"] = df["raw_label"].map(normalize_label_name)
    return df

def load_csv_test_set(data_path, text_column):
    df = pd.read_csv(data_path)
    required_columns = {text_column, "label"}
    missing_columns = required_columns - set(df.columns)
    if missing_columns:
        raise ValueError(f"{data_path} is missing required columns: {sorted(missing_columns)}")
    df[text_column] = df[text_column].astype(str).str.strip()
    df["label"] = df["label"].map(normalize_label_name)
    return df

def load_test_set(test_name, test_spec):
    data_path = test_spec["path"]
    if not data_path.exists():
        raise FileNotFoundError(f"Missing test set for {test_name}: {data_path}")

    if test_spec["kind"] == "tsv":
        df = load_original_tsv(data_path)
    else:
        df = load_csv_test_set(data_path, test_spec["text_column"])

    df = df.loc[df[test_spec["text_column"]].ne("")].copy().reset_index(drop=True)
    df["labels"] = df["label"].map(CANONICAL_LABEL2ID)
    return df

test_sets = {
    test_name: load_test_set(test_name, test_spec)
    for test_name, test_spec in TEST_SET_SPECS.items()
}

for test_name, df in test_sets.items():
    print(f"{test_name:>12}: {len(df):4d} rows")
    print(df["label"].value_counts().sort_index().to_string())
    print()

display(test_sets["localized"].head(3))


## Evaluation Helpers


In [ ]:
def to_python_scalars(value):
    if isinstance(value, dict):
        return {key: to_python_scalars(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_python_scalars(item) for item in value]
    if isinstance(value, tuple):
        return [to_python_scalars(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    return value

def calculate_classification_metrics(true_ids, predicted_ids):
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        true_ids, predicted_ids, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        true_ids, predicted_ids, average="weighted", zero_division=0
    )

    return {
        "accuracy": accuracy_score(true_ids, predicted_ids),
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
    }

def load_model_and_tokenizer(model_path):
    if not model_path.exists():
        raise FileNotFoundError(f"Missing saved model directory: {model_path}")
    tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    model.to(DEVICE)
    model.eval()
    return model, tokenizer

def predict_label_ids(model, tokenizer, texts):
    predicted_ids = []
    for start in range(0, len(texts), EVAL_BATCH_SIZE):
        batch_texts = texts[start : start + EVAL_BATCH_SIZE]
        encoded = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
        with torch.no_grad():
            logits = model(**encoded).logits
        model_predicted_ids = logits.argmax(dim=-1).cpu().tolist()
        for predicted_id in model_predicted_ids:
            model_label = model.config.id2label[int(predicted_id)]
            canonical_label = normalize_label_name(model_label)
            predicted_ids.append(CANONICAL_LABEL2ID[canonical_label])
    return np.array(predicted_ids)

def evaluate_model_on_test_set(model_name, model_spec, test_name, test_spec, test_df):
    model, tokenizer = load_model_and_tokenizer(model_spec["path"])
    texts = test_df[test_spec["text_column"]].tolist()
    true_ids = test_df["labels"].to_numpy()
    predicted_ids = predict_label_ids(model, tokenizer, texts)

    metrics = calculate_classification_metrics(true_ids, predicted_ids)
    report = classification_report(
        true_ids,
        predicted_ids,
        labels=list(range(len(CANONICAL_ID2LABEL))),
        target_names=[CANONICAL_ID2LABEL[idx] for idx in range(len(CANONICAL_ID2LABEL))],
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report).transpose()
    report_path = OUTPUT_DIR / f"{model_name}__{test_name}_classification_report.csv"
    report_df.to_csv(report_path)

    return {
        "model_key": model_name,
        "model_name": model_spec["display_name"],
        "test_key": test_name,
        "test_name": test_spec["display_name"],
        "num_rows": len(test_df),
        **to_python_scalars(metrics),
        "classification_report_path": str(report_path),
    }


## Run Evaluation


In [ ]:
evaluation_rows = []

for model_name, model_spec in MODEL_SPECS.items():
    for test_name, test_spec in TEST_SET_SPECS.items():
        print(f"Evaluating {model_spec['display_name']} on {test_spec['display_name']}...")
        row = evaluate_model_on_test_set(
            model_name=model_name,
            model_spec=model_spec,
            test_name=test_name,
            test_spec=test_spec,
            test_df=test_sets[test_name],
        )
        evaluation_rows.append(row)

results_df = pd.DataFrame(evaluation_rows)
results_df = results_df.sort_values(["model_key", "test_key"]).reset_index(drop=True)

summary_csv_path = OUTPUT_DIR / "ablation_summary_metrics.csv"
summary_json_path = OUTPUT_DIR / "ablation_summary_metrics.json"
results_df.to_csv(summary_csv_path, index=False)
with summary_json_path.open("w", encoding="utf-8") as fp:
    json.dump(to_python_scalars(results_df.to_dict(orient="records")), fp, indent=2)

display(results_df[["model_name", "test_name", "accuracy", "f1_macro", "f1_weighted"]])
print(f"Saved summary CSV to: {summary_csv_path}")
print(f"Saved summary JSON to: {summary_json_path}")


## Comparison Views


In [ ]:
f1_macro_pivot = results_df.pivot(index="model_name", columns="test_name", values="f1_macro")
accuracy_pivot = results_df.pivot(index="model_name", columns="test_name", values="accuracy")

print("Macro-F1")
display(f1_macro_pivot)

print("Accuracy")
display(accuracy_pivot)
